# 01 — Python from zero

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 1, Lesson 2 (60 minutes).

Goal: Calculate S0003 version 1 sales before returns using Python.

Before running: Dataset uploaded; assigned configuration filled in. The calculation steps also run in ordinary Python.

Writes: No files or tables.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## 01-01: Give a value a name

**Core concept Vibe:** A variable holds a value we can reuse.

**Build Buddy action:** Create variables for the sale ID and its quantity.

**Expected checkpoint:** S0003 and quantity 3.


In [0]:
sale_id = 'S0003'
quantity = 3

print(sale_id)
print(quantity)

assert sale_id == 'S0003' and quantity == 3

S0003
3


## 01-02: Represent money and a percentage

**Core concept Vibe:** Text, whole numbers, and decimal numbers have different uses.

**Build Buddy action:** Represent the unit price and discount. Decimal preserves exact money values.

**Expected checkpoint:** INR 150.00 per Tea 250 g pack; 10 means 10 percent.


In [0]:
from decimal import Decimal, ROUND_HALF_UP

unit_price = Decimal('150.00')
discount_pct = Decimal('10')

print('Price per pack:', unit_price)
print('Discount percentage points:', discount_pct)

assert unit_price == Decimal('150.00') and discount_pct == 10


Price per pack: 150.00
Discount percentage points: 10


## 01-03: Multiply quantity by price

**Core concept Vibe:** Multiplication calculates the amount before discounts.

**Build Buddy action:** Calculate gross sales.

**Expected checkpoint:** Gross sales = INR 450.00.


In [0]:
gross_sales = quantity * unit_price

print('Gross sales:', gross_sales)

assert gross_sales == Decimal('450.00')

Gross sales: 450.00


## 01-04: Calculate the discount

**Core concept Vibe:** A percentage means a number out of 100.

**Build Buddy action:** Convert 10 percentage points to a fraction and calculate the discount amount.

**Expected checkpoint:** Discount fraction = 0.1; discount amount = INR 45.00.


In [0]:
discount_fraction = discount_pct / 100

discount_amount = (gross_sales * discount_fraction).quantize(
    Decimal('0.01'),
    rounding=ROUND_HALF_UP
)

print('Discount fraction:', discount_fraction)
print('Discount amount:', discount_amount)

assert discount_fraction == Decimal('0.1')
assert discount_amount == Decimal('45.00')

Discount fraction: 0.1
Discount amount: 45.00


## 01-05: Subtract the discount

**Core concept Vibe:** Net sales is sales after discounts and before returns.

**Build Buddy action:** Calculate and display net sales.

**Expected checkpoint:** S0003 version 1 net sales = INR 405.00.


In [0]:
net_sales = gross_sales - discount_amount

print(f'{sale_id}: net sales before returns = INR {net_sales:.2f}')

assert net_sales == Decimal('405.00')

S0003: net sales before returns = INR 405.00


## 01-06: Group related values

**Core concept Vibe:** A dictionary gives names to several values in one record.

**Build Buddy action:** Create a dictionary describing this sale calculation.

**Expected checkpoint:** The dictionary holds the same values; no new sale is invented.


In [0]:
sale = {'sale_id': sale_id, 'quantity': quantity, 'unit_price': unit_price, 'discount_pct': discount_pct, 'net_sales': net_sales}
print(sale['sale_id'], sale['net_sales'])
assert sale['quantity'] == 3

S0003 405.00


## 01-07: Verify against the real file

**Core concept Vibe:** An answer should be traceable to its source record.

**Build Buddy action:** Read S0003 from the initial CSV and check the values used above.

**Expected checkpoint:** The manually entered values exactly match the released CSV.


In [0]:
from pathlib import Path
import csv
with (Path(DATASET_ROOT) / 'data/sales_batch_01.csv').open(encoding='utf-8', newline='') as stream:
    rows = list(csv.DictReader(stream))
matches = [row for row in rows if row['sale_id'] == sale_id]
assert len(matches) == 1
source_sale = matches[0]
assert source_sale['source_row_id'] == 'B01-003'
assert int(source_sale['quantity']) == quantity
assert Decimal(source_sale['unit_price']) == unit_price
assert Decimal(source_sale['discount_pct']) == discount_pct
assert source_sale['record_version'] == '1'
print('PASS: calculation matches B01-003 / S0003 version 1.')

PASS: calculation matches B01-003 / S0003 version 1.


## Explain it before moving on

What is a variable? Why divide discount_pct by 100? Why are net sales different from profit? Explain why this notebook reports version 1 and does not yet apply the Day 2 correction.

Suggested pacing: 10 minutes for variables, 10 for data types and money, 20 for arithmetic, 10 for the dictionary and source lookup, 10 for practice and explanation. Keep the original cells intact for exact checks; use a scratch cell for other numbers.
